# LAB 01 [SOLUTION] — Platform Exploration

**Databricks Fundamentals | ~60 min | After: M01 Platform & Workspace**

---

## Scenario

You just joined the Data Engineering team at **RetailHub**. On day one you need to:
1. Confirm your environment is correctly set up
2. Navigate the workspace and understand the data catalog structure
3. Write and run your first code cells
4. Get comfortable with the key tools: magic commands, dbutils, Unity Catalog

## Task Map

| Task | Topic | Key Concepts |
|------|-------|--------------|
| 01 | Setup & environment verification | `%run`, variables, SparkSession |
| 02 | Compute navigation — clusters & Serverless | Cluster types, DBU |
| 03 | First Python & SQL cells | `spark.version`, `current_user()` |
| 04 | Magic commands in practice | `%sql`, `%fs`, `%sh`, `%pip` |
| 05 | dbutils — your Swiss Army knife | `dbutils.fs`, `dbutils.widgets` |
| 06 | Unity Catalog — explore the structure | `SHOW CATALOGS`, `SHOW SCHEMAS` |
| 07 | Your first Delta table | `CREATE TABLE`, `INSERT`, `SELECT` |
| 08 | Catalog Explorer — GUI verification | Lineage, Sample Data |
| 09 | Cost check — where to look | Account Console, Compute metrics |

> **Tip:** Read every comment before running a cell. If you finish early, check the **Bonus Challenges** section at the bottom.

---
## Task 01 — Setup & Environment Verification

Every notebook starts by running `00_setup`, which:
- identifies your user
- validates your catalog in Unity Catalog
- exports variables: `CATALOG`, `BRONZE_SCHEMA`, `SILVER_SCHEMA`, `GOLD_SCHEMA`, `DATASET_PATH`

In [ ]:
%run ../setup/00_setup

In [ ]:
# Verify that all variables were loaded correctly
print(f"User:        {raw_user}")
print(f"Catalog:     {CATALOG}")
print(f"Bronze:      {BRONZE_SCHEMA}")
print(f"Silver:      {SILVER_SCHEMA}")
print(f"Gold:        {GOLD_SCHEMA}")
print(f"Datasets:    {DATASET_PATH}")
print()
print(f"Spark version:  {spark.version}")
print(f"Spark app name: {spark.sparkContext.appName}")

#  Check — all variables must be set
assert CATALOG.startswith("retailhub_"), f"Unexpected catalog name: {CATALOG}"
assert DATASET_PATH != "", "DATASET_PATH is empty — re-run 00_setup"
print("\n Environment OK")

**Question:** What Spark version is on your cluster? Look it up in the Databricks docs — which DBR (Databricks Runtime) version does it correspond to?

---
## Task 02 — Compute Navigation

**Do this in the GUI (no code):**

1. Click **Compute** in the left menu.
2. Find the cluster you are using — check:
   - Cluster type (All-Purpose / Job / Serverless?)
   - Databricks Runtime version (DBR)
   - Number of workers (min/max if autoscaling)
   - Is Photon enabled?
3. Open **Configuration** → review Spark config and Environment variables.
4. Click **Event Log** — what do you see in the startup logs?

Note your answers below:

In [ ]:
# Programmatic verification — same info you found in the GUI
dbr          = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion", "unknown")
photon       = spark.conf.get("spark.databricks.photon.enabled", "false")
cluster_name = spark.conf.get("spark.databricks.clusterUsageTags.clusterName", "unknown")
cluster_type = spark.conf.get("spark.databricks.clusterUsageTags.clusterSku", "unknown")

print(f"Cluster name:  {cluster_name}")
print(f"Cluster type:  {cluster_type}")
print(f"DBR version:   {dbr}")
print(f"Photon:        {photon}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

---
## Task 03 — First Python & SQL Cells  `[TODO]`

Run the cells below. Read the output and understand what each one does.

In [ ]:
# Python in a Databricks notebook — normal variables and functions
message = f"Hello, {raw_user}! Welcome to Databricks."
print(message)

# SparkSession is always available as 'spark'
print(f"\nSparkContext master: {spark.sparkContext.master}")
print(f"Shuffle partitions:  {spark.conf.get('spark.sql.shuffle.partitions')}")

In [ ]:
%sql
-- SQL runs directly thanks to the %sql magic command
SELECT
    current_user()    AS my_user,
    current_catalog() AS my_catalog,
    current_schema()  AS my_schema,
    current_date()    AS today,
    version()         AS spark_version

In [ ]:
# Task 03 [SOLUTION]
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

df_cities = spark.createDataFrame(
    [
        ('Warsaw',    'Poland',      1800000),
        ('Berlin',    'Germany',     3700000),
        ('Paris',     'France',      2100000),
        ('Amsterdam', 'Netherlands',  900000),
    ],
    ['city', 'country', 'population']
)
display(df_cities)

In [ ]:
#  Check Task 03
assert df_cities is not None, "df_cities is None — did you implement the TODO?"
assert df_cities.count() >= 3, f"Expected at least 3 rows, got {df_cities.count()}"
assert "city"       in df_cities.columns, "Missing column: city"
assert "population" in df_cities.columns, "Missing column: population"
print(f" Task 03 passed — {df_cities.count()} cities in DataFrame")
display(df_cities)

---
## Task 04 — Magic Commands in Practice  `[TODO]`

Magic commands start with `%` (single cell) or `%%` (whole cell).

In [ ]:
# %fs — file system operations (shorthand for dbutils.fs)
%fs ls /databricks-datasets/

In [ ]:
# %sh — shell commands (Linux/bash)
%sh echo "Hostname: $(hostname)" && python3 --version

In [ ]:
# %pip — install a library into the current notebook session
%pip install faker --quiet

In [ ]:
# TODO: Use the installed Faker library to generate 5 random full names
# Requirements:
#   - import Faker
#   - create a Faker() instance
#   - generate a Python list called `fake_names` with exactly 5 names
#   - print each name

from faker import Faker

# Your code here:
fake_names = []  # replace with your implementation

In [ ]:
#  Check Task 04
assert isinstance(fake_names, list), "fake_names must be a Python list"
assert len(fake_names) == 5,         f"Expected 5 names, got {len(fake_names)}"
assert all(isinstance(n, str) for n in fake_names), "All names must be strings"
print(f" Task 04 passed — generated {len(fake_names)} names:")
for n in fake_names:
    print(f"   {n}")

In [ ]:
# List the contents of your data Volume
volume_path = DATASET_PATH
print(f"Volume path: {volume_path}")
display(dbutils.fs.ls(volume_path))

---
## Task 05 — dbutils — Your Swiss Army Knife

`dbutils` is a utility module available in every Databricks notebook:

| Module | Purpose |
|--------|---------|
| `dbutils.fs` | File operations (ls, cp, mv, rm, mkdirs) |
| `dbutils.widgets` | Interactive parameter inputs in a notebook |
| `dbutils.notebook` | Call other notebooks, pass results |
| `dbutils.secrets` | Securely read secrets (API keys, passwords) |

In [ ]:
# dbutils.fs — file system operations
# Verify the customers.csv file exists
csv_path = f"{DATASET_PATH}/customers/customers.csv"

try:
    files = dbutils.fs.ls(f"{DATASET_PATH}/customers/")
    for f in files:
        print(f"  {f.name:40s}  {f.size:>10,} bytes")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# dbutils.widgets — interactive parameter inputs
# After running this cell, a text box appears at the top of the notebook
dbutils.widgets.text("city_filter", "London", "Filter by city")

# Read the widget value
city = dbutils.widgets.get("city_filter")
print(f"Selected filter: {city}")

In [ ]:
# dbutils.help() — full documentation for all modules
dbutils.help()

In [ ]:
# Clean up widgets after use
dbutils.widgets.removeAll()

---
## Task 06 — Unity Catalog — Explore the Structure  `[TODO]`

Explore the Unity Catalog hierarchy both programmatically and in the GUI.

**Structure reminder:**
```
Metastore
└── Catalog (retailhub_<your_username>)
    ├── Schema: bronze
    ├── Schema: silver
    └── Schema: gold
```

In [ ]:
# List available catalogs
print("=== CATALOGS ===")
display(spark.sql("SHOW CATALOGS"))

In [ ]:
# List schemas in your catalog
print(f"=== SCHEMAS IN {CATALOG} ===")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

In [ ]:
# Check how many tables exist in each schema (should be empty at this point)
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").count()
    print(f"  {CATALOG}.{schema}: {tables} table(s)")

In [ ]:
# TODO: Query Unity Catalog metadata using information_schema
# List all schemas in YOUR catalog using:
#   SELECT * FROM system.information_schema.schemata
#   WHERE catalog_name = '<your_catalog>'
#
# Store the result in a variable called `df_schemas`

# Your code here:
df_schemas = None  # replace with spark.sql(...)

In [ ]:
#  Check Task 06
assert df_schemas is not None, "df_schemas is None — did you implement the TODO?"
schema_count = df_schemas.count()
assert schema_count >= 3, f"Expected at least 3 schemas (bronze/silver/gold), got {schema_count}"
schema_names = [r[0] for r in df_schemas.select("schema_name").collect()]
assert "bronze" in schema_names, "Schema 'bronze' not found"
assert "silver" in schema_names, "Schema 'silver' not found"
assert "gold"   in schema_names, "Schema 'gold' not found"
print(f" Task 06 passed — found {schema_count} schemas: {schema_names}")

**Do this in the GUI:**
1. Click **Catalog** in the left menu.
2. Find your catalog (`retailhub_<your_username>`).
3. Expand schemas `bronze`, `silver`, `gold`.
4. Click one schema → check the **Details** tab.
5. Also explore `samples` → `nyctaxi` → table `trips` → **Sample Data** tab.

---
## Task 07 — Your First Delta Table  `[TODO]`

Create your first Delta table in your catalog. This is the fundamental operation in Databricks.

In [ ]:
# Define the full table name (3-level namespace)
table_name = f"{CATALOG}.{BRONZE_SCHEMA}.my_first_table"
print(f"Creating table: {table_name}")

In [ ]:
# Create a table using SQL and insert sample data
spark.sql(f"""
    CREATE OR REPLACE TABLE {table_name} (
        id        INT,
        name      STRING,
        role      STRING,
        joined_at DATE
    )
    USING DELTA
    COMMENT 'My first Delta table — Lab 01'
""")

spark.sql(f"""
    INSERT INTO {table_name} VALUES
        (1, 'Anna Kowalska',   'Data Engineer',  '2024-01-10'),
        (2, 'Piotr Nowak',     'Data Analyst',   '2024-03-22'),
        (3, 'Maria Wisniewska','BI Specialist',  '2024-06-05'),
        (4, 'Jakub Zajac',     'Data Engineer',  '2025-01-15')
""")

print("Table created and populated.")

In [ ]:
# Read and display the table
display(spark.table(table_name))

In [ ]:
# TODO: Retrieve the table history — how many versions exist?
# Use: DESCRIBE HISTORY <table_name>
# Store the result as: history_df

history_df = None  # replace with: spark.sql(f"DESCRIBE HISTORY {table_name}")

In [ ]:
#  Check — Task 07a: DESCRIBE HISTORY
assert history_df is not None, "history_df is None — did you run DESCRIBE HISTORY?"
assert history_df.count() >= 2, f"Expected at least 2 versions (CREATE + INSERT), got {history_df.count()}"
assert "operation" in history_df.columns, "Missing 'operation' column in history result"
print(f" Task 07a passed — table has {history_df.count()} version(s) in history")
display(history_df)

In [ ]:
# TODO: Write a SQL query that counts people per role (GROUP BY role)
# Use spark.sql() to run the query
# Store the result as: df_by_role

df_by_role = None  # replace with: spark.sql(f"SELECT role, COUNT(*) AS count FROM {table_name} GROUP BY role")

In [ ]:
#  Check — Task 07b: GROUP BY role
assert df_by_role is not None, "df_by_role is None — did you run the GROUP BY query?"
assert df_by_role.count() >= 1, f"Expected at least 1 group, got {df_by_role.count()}"
assert "role" in df_by_role.columns, "Missing 'role' column in result"
print(f" Task 07b passed — found {df_by_role.count()} distinct role(s)")
display(df_by_role)

---
## Task 08 — Catalog Explorer — GUI Verification

After creating the table in Task 07, return to the GUI and inspect it in Catalog Explorer.

**Steps:**
1. Click **Catalog** → your catalog → `bronze` → `my_first_table`.
2. Check the **Overview** tab — is the `COMMENT` visible?
3. Open the **Sample Data** tab — does the data match what you inserted?
4. Open the **History** tab — how many versions does the table have? What was recorded after the INSERT?
5. Open the **Lineage** tab — still empty, but revisit it after LAB 02.

> **Why it matters?** Catalog Explorer is a GUI on top of Unity Catalog — auditors and data owners use it to inspect data without writing any code.

---
## Task 09 — Cost Check — Where to Look  `[TODO]`

Every cluster run costs DBUs. Learn where to monitor usage and estimated costs.

In [ ]:
# System Tables — programmatic cost monitoring
# The system.billing.usage table contains DBU usage logs (if available in your workspace)
try:
    usage_df = spark.sql("""
        SELECT
            usage_date,
            sku_name,
            ROUND(SUM(usage_quantity), 4) AS total_dbu
        FROM system.billing.usage
        WHERE usage_date >= current_date() - 7
        GROUP BY usage_date, sku_name
        ORDER BY usage_date DESC, total_dbu DESC
        LIMIT 20
    """)
    display(usage_df)
except Exception as e:
    print(f"System tables not available in this workspace: {e}")
    print("Use Account Console → Cost Analysis in your browser.")

**Check in the GUI:**
1. **Compute** → your cluster → **Metrics** tab (CPU, memory, DBU/hour)
2. The trainer may show **Account Console** (account.cloud.databricks.com) → **Cost Analysis**

> **Rule of thumb:** An All-Purpose cluster costs money the entire time it is running — even when idle. Always set **auto-termination** (30 min idle) to avoid unexpected charges!

---
## Cleanup

Optional — drop the test table to keep your catalog tidy.

In [ ]:
# Uncomment and run only if you want to remove the test table
# spark.sql(f"DROP TABLE IF EXISTS {table_name}")
# print(f"Dropped: {table_name}")

---
##  Bonus Challenges

Finished early? Try these extra tasks:

### Bonus 1 — Volumes in practice
Create a new folder inside your Volume and copy a file into it:
```python
dbutils.fs.mkdirs(f"{DATASET_PATH}/my_output")
dbutils.fs.cp("<path_to_csv_file>", f"{DATASET_PATH}/my_output/test.csv")
dbutils.fs.ls(f"{DATASET_PATH}/my_output")
```

### Bonus 2 — Temp view + SQL
Load `my_first_table` into a DataFrame, register it as a temp view, then query it with SQL:
```python
df = spark.table(table_name)
df.createOrReplaceTempView("team")
# then: spark.sql("SELECT * FROM team WHERE role = 'Data Engineer'")
```

### Bonus 3 — Table metadata
Inspect the technical details of your Delta table:
```sql
DESCRIBE DETAIL <your_table>
DESCRIBE EXTENDED <your_table>
```
How many Parquet files does your table have? How many bytes does it occupy?

### Bonus 4 — Built-in datasets
Databricks ships sample datasets for learning. Explore them:
```python
display(spark.table("samples.nyctaxi.trips").limit(5))
spark.table("samples.tpch.orders").printSchema()
```

---
## Lab 01 Summary

After this lab you should be able to:

| Skill | Status |
|-------|--------|
| Run the setup notebook and read environment variables | [ ] |
| Verify the cluster type and parameters in the GUI | [ ] |
| Write Python and SQL cells in the same notebook | [ ] |
| Use magic commands: `%sql`, `%fs`, `%sh`, `%pip` | [ ] |
| Use `dbutils.fs` and `dbutils.widgets` | [ ] |
| Navigate Unity Catalog (catalogs, schemas, tables) | [ ] |
| Create and populate your first Delta table | [ ] |
| Find the table in Catalog Explorer (GUI) | [ ] |
| Know where to check costs (Compute / Account Console) | [ ] |

> **Next:** LAB 02 — ELT Pipeline (Bronze → Silver ingestion)